In [0]:
landing_path = "/Volumes/meteo/default/apilanding/landing/"
checkpoint_bronze = "/Volumes/meteo/default/apilanding/checkpoints/bronze/"
checkpoint_silver = "/Volumes/meteo/default/apilanding/checkpoints/silver/"

In [0]:
import requests
import json
from datetime import datetime

cities = [
    {"city": "Krakow", "country": "Poland", "latitude": 50.0647, "longitude": 19.9450},
    {"city": "Warsaw", "country": "Poland", "latitude": 52.2297, "longitude": 21.0122},
    {"city": "Berlin", "country": "Germany", "latitude": 52.5200, "longitude": 13.4050},
    {"city": "London", "country": "UK", "latitude": 51.5072, "longitude": -0.1276}
]

def fetch_weather_to_landing():
    run_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    for c in cities:
        url = (
            "https://api.open-meteo.com/v1/forecast"
            f"?latitude={c['latitude']}"
            f"&longitude={c['longitude']}"
            "&current=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation"
        )

        response = requests.get(url)
        response.raise_for_status()

        data = response.json()

        record = {
            "city": c["city"],
            "country": c["country"],
            "latitude": c["latitude"],
            "longitude": c["longitude"],
            "ingestion_time_utc": datetime.utcnow().isoformat(),
            "api_response": data
        }

        file_name = f"{landing_path}/{c['city']}_{run_ts}.json"

        dbutils.fs.put(file_name, json.dumps(record), overwrite=True)

    print(f"Weather files created for run: {run_ts}")


In [0]:
from pyspark.sql.types import *


def get_weather_schema():
    """
    Returns the schema for Open-Meteo weather JSON files.
    """

    schema = StructType([
        StructField("city", StringType()),
        StructField("country", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("ingestion_time_utc", StringType()),

        StructField("api_response", StructType([
            StructField("latitude", DoubleType()),
            StructField("longitude", DoubleType()),
            StructField("generationtime_ms", DoubleType()),
            StructField("utc_offset_seconds", IntegerType()),
            StructField("timezone", StringType()),
            StructField("timezone_abbreviation", StringType()),
            StructField("elevation", DoubleType()),

            StructField("current_units", MapType(StringType(), StringType())),

            StructField("current", StructType([
                StructField("time", StringType()),
                StructField("interval", IntegerType()),
                StructField("temperature_2m", DoubleType()),
                StructField("relative_humidity_2m", DoubleType()),
                StructField("wind_speed_10m", DoubleType()),
                StructField("precipitation", DoubleType())
            ]))
        ]))
    ])

    return schema


def read_weather_stream():
    """
    Reads weather JSON files from the landing path as a streaming DataFrame.

    Parameters:
        spark: SparkSession
        landing_path: Path where JSON weather files are stored

    Returns:
        Streaming DataFrame
    """

    schema = get_weather_schema()

    weather_stream_df = (
        spark.readStream
        .format("json")
        .schema(schema)
        .load(landing_path)
    )

    return weather_stream_df